# 02 - Continual learning with replay and incremental polynomial expansion

Experiences are processed one at a time. At experience *t* the learner may use only the current experience's training
samples, its replay buffer (training samples of earlier experiences), and its current parameters. Evaluation samples are
never used for learning or replay. The polynomial degree grows by one per experience (default), following the **expansion
invariant**: old coefficients are copied exactly, new coefficients start at zero, and `f_{d+1}(x) = f_d(x)` is verified (scores and
probabilities, on training, evaluation, random and out-of-region inputs) before any training.

Nothing here assumes the method works. Compare the strategies at the end before drawing any conclusion.

## Configuration

In [ ]:
SEED = 0
N_TRAIN, N_TEST, NOISE = 200, 500, 0.0
STRATEGY = "ordinary"  # ordinary_no_replay | ordinary | frozen_old | small_old_lr | strategy_a_all | strategy_a_new | strategy_b_all | strategy_b_new
STOP_AFTER = (
    None  # e.g. 3 stops after experience 3 so intermediate behaviour can be inspected
)
DEGREES = None  # None -> 1, 2, ..., 10 ; or give an explicit non-decreasing list
LR, EPOCHS = 0.5, 300
M_CURRENT, M_REPLAY, CAPACITY, REPLACE = (
    100,
    100,
    100,
    False,
)  # balanced replay: m per experience
SHOW_STEPS = [
    3,
    6,
    10,
]  # experiences (beyond 1 and 2, shown above) for which the before/after figure is drawn

In [ ]:
import importlib.util
import pathlib
import sys

if importlib.util.find_spec("polynomial_cl") is None:  # not pip-installed: use src/
    sys.path.insert(0, str(pathlib.Path("..").resolve() / "src"))
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams.update({"figure.dpi": 80})
from polynomial_cl import (
    STRATEGIES,
    make_config,
    make_experiences,
    metrics,
    run_continual,
    run_seeds,
)
from polynomial_cl import plotting as P

In [ ]:
exps = make_experiences(seed=SEED, n_train=N_TRAIN, n_test=N_TEST, noise=NOISE)
cfg = make_config(
    STRATEGY,
    lr=LR,
    epochs=EPOCHS,
    m_current=M_CURRENT,
    m_replay=M_REPLAY,
    capacity=CAPACITY,
    replace=REPLACE,
    seed=SEED,
    stop_after=STOP_AFTER,
    degrees=DEGREES,
)
fig, axs = plt.subplots(1, 2, figsize=(9, 4.2))
P.plot_ground_truth(axs[0], title="Ground truth")
P.plot_ground_truth(axs[1], title="Experiences = vertical strips (training samples)")
for e in exps:
    P.plot_samples(axs[1], e.train.x, e.train.y, s=6)
    axs[1].axvline(e.window[0], color="k", lw=0.3)
plt.tight_layout()
plt.show()
result = run_continual(exps, cfg)
print(cfg)

## Stage A - initial model (experience 1)

In [ ]:
r1 = result.records[0]
print("degree:", r1.degree_after, "| features:", r1.feature_names)
print(f"training loss: {r1.loss_history[0]:.4f} -> {r1.loss_history[-1]:.4f}")
print(f"accuracy on experience 1 (fixed test set): {r1.acc_current:.3f}")
stage_a = {
    "scores": r1.model_after.score(exps[0].test.x),
    "labels": r1.model_after.predict(exps[0].test.x),
    "coefficients": r1.model_after.w.copy(),
}  # saved for later comparison
print(P.format_coefficient_table(r1))
fig = P.plot_step(r1, exps)
plt.show()

## Stage B - experience 2: expand, verify the invariant, then train

The second panel (right after expansion, no training) must be identical to the first.

In [ ]:
r2 = result.records[1]
print(
    f"expansion degree {r2.degree_before} -> {r2.degree_after}:  max|score diff| = {r2.invariant_score_diff:.2e},  max|proba diff| = {r2.invariant_proba_diff:.2e}"
)
print("replay batch composition (samples drawn per experience):", r2.batch_counts)
print(P.format_coefficient_table(r2))
P.plot_step(r2, exps)
plt.show()
P.plot_coefficients(r2)
plt.show()

## Stage C - experiences 3 ... 10

In [ ]:
for t in SHOW_STEPS:
    if t - 1 < len(result.records):
        r = result.records[t - 1]
        print(
            f"--- experience {t}: degree {r.degree_before} -> {r.degree_after}, batch {r.batch_counts}"
        )
        print(
            f"    invariant: score diff {r.invariant_score_diff:.1e}, proba diff {r.invariant_proba_diff:.1e}"
        )
        P.plot_step(r, exps)
        plt.show()
        P.plot_coefficients(r)
        plt.show()

## Learning history and compact summary

In [ ]:
P.plot_learning_history(result)
plt.show()
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
P.plot_accuracy_matrix(result, axs[0])
last = result.records[-1].model_after
x_te = np.concatenate([e.test.x for e in exps[: len(result.records)]])
y_te = np.concatenate([e.test.y for e in exps[: len(result.records)]])
P.plot_correctness(
    last,
    axs[1],
    samples=(x_te, y_te),
    title="Final model: correct/incorrect regions + all evaluation samples",
)
plt.tight_layout()
plt.show()
print(metrics.format_table(metrics.summary_table(result)))

## Strategy comparison (single seed, identical data, evaluation sets, learning rate, epochs and replay budget)

| name | what changes |
|---|---|
| `ordinary_no_replay` | one global learning rate, **no replay** (isolates the effect of replay) |
| `ordinary` | one global learning rate, replay |
| `frozen_old` | existing coefficients frozen; only newly introduced coefficients train |
| `small_old_lr` | existing coefficients use 0.1 x the learning rate |
| `strategy_a_all` | `f = f_old + g`; g unconstrained over all coefficients. *Numerically identical to `ordinary`* (same function class, same start) - shown to make that explicit |
| `strategy_a_new` | `f = f_old + g`; g only over new coefficients (identical to `frozen_old`) |
| `strategy_b_all` | `f = f_old + g` with the constraint `g(x_ref) = 0` on reference inputs (3 buffered inputs per old experience); g over all coefficients |
| `strategy_b_new` | as above, g only over new coefficients |

Preserving `f_old` at a finite set of reference points does **not** preserve it elsewhere; the columns below measure that directly.

In [ ]:
names = list(STRATEGIES)


def cfg_for(n, **kw):
    shared = {
        "lr": LR,
        "epochs": EPOCHS,
        "m_current": M_CURRENT,
        "m_replay": M_REPLAY,
        "capacity": CAPACITY,
        "replace": REPLACE,
    }
    shared = {
        k: v for k, v in shared.items() if k not in STRATEGIES[n]
    }  # presets keep their defining fields
    return make_config(n, **shared, **kw)


runs = {n: run_continual(exps, cfg_for(n, seed=SEED)) for n in names}
rows = []
for n, r in runs.items():
    last = r.records[-1]
    rows.append(
        {
            "strategy": n,
            "final_mean_acc": last.mean_acc,
            "final_acc_current": last.acc_current,
            "final_acc_E1": r.acc_matrix[-1, 0],
            "final_mean_forgetting": last.mean_forgetting,
            "worst_forgetting_ever": max(x.max_forgetting for x in r.records),
            "mean_max_old_score_change": float(
                np.mean([x.max_old_score_change for x in r.records[1:]])
            ),
            "mean_old_flip_frac": float(
                np.mean([x.old_flip_fraction for x in r.records[1:]])
            ),
            "max_frozen_change": max((x.frozen_max_change for x in r.records if x.frozen_max_change is not None), default=float("nan")),
        }
    )
print(metrics.format_table(rows))
plt.figure(figsize=(7, 4))
for n, r in runs.items():
    plt.plot(range(1, 11), [x.mean_acc for x in r.records], marker="o", ms=3, label=n)
plt.xlabel("experience")
plt.ylabel("mean accuracy on observed experiences")
plt.legend(fontsize=7)
plt.ylim(0.5, 1)
plt.show()

### Ablation: does the incremental degree schedule matter?

Same data, replay and budget, but the model has degree 10 from experience 1 (all 66 coefficients exist and are trainable
from the start) instead of growing 1, 2, ..., 10. Five seeds.

In [ ]:
abl = {}
for label, deg in (("grow 1..10", None), ("fixed degree 10", [10] * 10)):
    rs = run_seeds(
        [0, 1, 2, 3, 4],
        cfg_for("ordinary", degrees=deg),
        {"n_train": N_TRAIN, "n_test": N_TEST, "noise": NOISE},
    )
    abl[label] = rs
    print(
        f"{label:16s} final mean acc {np.mean([r.records[-1].mean_acc for r in rs]):.3f} +/- {np.std([r.records[-1].mean_acc for r in rs]):.3f} | "
        f"final mean forgetting {np.mean([r.records[-1].mean_forgetting for r in rs]):.3f} | "
        f"mean current-experience acc over the run {np.mean([[x.acc_current for x in r.records] for r in rs]):.3f}"
    )

### Strategy B inspection

`ref_score_change` (change at the reference inputs) should be ~1e-15 by construction. `old_score_change` is measured on the
*whole* old evaluation sets, which are not the reference inputs. `free dims` is the number of coefficient directions the
constraint leaves for learning (0 means g cannot learn at all, and that is reported as is).

In [ ]:
for n in ("strategy_b_all", "strategy_b_new"):
    print("==", n)
    print(
        metrics.format_table(
            [
                {
                    "exp": r.t + 1,
                    "degree": r.degree_after,
                    "free_dims": r.null_dim if r.null_dim is not None else "-",
                    "acc_current": r.acc_current,
                    "mean_acc": r.mean_acc,
                    "ref_score_change": r.ref_score_change
                    if r.ref_score_change is not None
                    else float("nan"),
                    "old_score_change(eval sets)": r.max_old_score_change,
                    "old_flip_frac": r.old_flip_fraction,
                    "frozen_change": r.frozen_max_change if r.frozen_max_change is not None else float("nan"),
                }
                for r in runs[n].records
            ],
            "{:.2g}",
        )
    )
P.plot_learning_history(runs["strategy_b_all"])
plt.show()
P.plot_step(runs["strategy_b_all"].records[-1], exps)
plt.show()

## Multiple seeds

Seeds change the training samples, replay selection and shuffling (evaluation sets also change with the seed but are
fixed within a seed). Values are mean +/- standard deviation over seeds.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]


def mean_std(runs_, getter):
    vals = [getter(r) for r in runs_]
    return np.mean(vals), np.std(vals)


rows = []
for n in names:
    base = cfg_for(n)
    rs = run_seeds(SEEDS, base, {"n_train": N_TRAIN, "n_test": N_TEST, "noise": NOISE})
    m1, s1 = mean_std(rs, lambda r: r.records[-1].mean_acc)
    m2, s2 = mean_std(rs, lambda r: r.records[-1].acc_current)
    m3, s3 = mean_std(rs, lambda r: r.records[-1].mean_forgetting)
    m4, s4 = mean_std(rs, lambda r: r.acc_matrix[-1, 0])
    rows.append(
        {
            "strategy": n,
            "final_mean_acc": f"{m1:.3f}+/-{s1:.3f}",
            "final_acc_current": f"{m2:.3f}+/-{s2:.3f}",
            "final_acc_E1": f"{m4:.3f}+/-{s4:.3f}",
            "final_mean_forgetting": f"{m3:.3f}+/-{s3:.3f}",
        }
    )
print(metrics.format_table(rows))